# 基于MindSpore的Qwen2.5-0.5B模型peft微调任务

## 案例介绍

本实验以应用于MRPC（Microsoft Research Paraphrase Corpus）语义匹配任务为例，展示了如何使用MindSpore框架和MindNLP库，对Qwen2.5-0.5B模型进行参数高效微调（PEFT）。

MRPC任务旨在判断两个句子在语义上是否等价（即是否为同义句），属于自然语言处理中的经典文本分类问题。通过此案例，您可以学习如何利用PEFT技术（例如LoRA），在计算资源受限的情况下，高效地使大语言模型适应特定下游任务。

通过完成本案例，您将能够：

1.掌握在MindSpore框架下使用PEFT技术微调大语言模型的基本流程。<br>
2.理解如何将预训练大模型适配到具体的文本分类任务。<br>
3.获得在有限算力下有效定制大模型的实际经验。<br>

接下来，您可以在此基础上继续构建PEFT模型，设置训练循环，并最终评估模型在MRPC测试集上的性能。

## 模型简介

Qwen2.5-0.5B是阿里巴巴通义千问团队开发的轻量级大语言模型，参数量约5亿。它采用先进的Transformer解码器架构，集成了GQA、RoPE、SwiGLU等关键技术，支持高达128K令牌的上下文长度和多语言处理能力。相比前代版本，其在代码和数学能力上提升显著，并优化了对结构化数据的理解与生成。得益于小巧的体积和高效的架构，该模型非常适合在手机、笔记本电脑等资源受限的设备上部署，为边缘计算和轻量级AI应用提供了良好基础。

## 环境准备

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [1]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

其他场景可参考[MindSpore安装指南](https://www.mindspore.cn/install)与[MindSpore NLP安装指南](https://github.com/mindspore-lab/mindnlp?tab=readme-ov-file#installation)进行环境搭建。

## 数据加载与预处理

### 数据集加载

mrpc_dataset

MRPC数据集，全称为Microsoft Research Paraphrase Corpus（微软研究院释义语料库），是一个用于NLP的句对相似性判断任务中性能评估的数据集。
MRPC数据集包含了大量从新闻、网页和论坛中收集的英文句子对。每个句子对都有一个人工标注的二元标签：0表示两句话不相似，1表示它们相似。


In [2]:
import mindnlp
from datasets import load_dataset
import numpy as np

mrpc_dict = load_dataset("SetFit/mrpc")   # 如果本地未下载会先下载，若已下载则会直接加载

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress 

In [3]:
# 查看原始数据集的结构
print("数据集结构:", mrpc_dict)

# 查看训练集的第一条原始数据
original_example = mrpc_dict['train'][0]
print("\n原始数据第一条:")
print(original_example)

数据集结构: DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 408
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'label_text'],
        num_rows: 1725
    })
})

原始数据第一条:
{'text1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'text2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0, 'label_text': 'equivalent'}


此次实验的指定的一些超参数

In [4]:
class Args:
    output_dir="./peft_model/mrpc_IA3"   # The output directory where the model checkpoints will be written.
    per_device_train_batch_size = 1  # Batch size per GPU/CPU for training
    per_device_eval_batch_size = 1
    model_name_or_path = "Qwen/Qwen2.5-0.5B"   
    num_train_epochs = 5
    learning_rate = 2e-5    # learning rate
    weight_decay = 0.01
    max_length = 256
    debug = True
    is_lora = True

args = Args()

### 数据预处理

在本案例中我们对数据使用GPT2的词汇表对数据集的样本特征进行token转换。

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
special_tokens_dict = {
    "bos_token": "<bos>",
    "eos_token": "<eos>",
    "pad_token": "<pad>",
}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(num_added_toks)

3


In [6]:
def tokenize_function(example):
    # 分词器自动处理句子对、添加 [CLS]、[SEP]，并生成 token_type_ids
    return tokenizer(example["text1"], example["text2"], truncation=True, padding="max_length",max_length=args.max_length)

In [7]:
tokenized_datasets = mrpc_dict.map(tokenize_function, batched=True)

# 将标签列重命名为 'labels'（这是大多数 transformers 模型预期的名称）
tokenized_datasets = tokenized_datasets.map(lambda examples: {"labels": examples["label"]}, batched=True)

# 移除模型不接受的列
tokenized_datasets = tokenized_datasets.remove_columns(["text1", "text2", "idx", "label"])

# 设置数据集返回格式为 PyTorch 张量
tokenized_datasets.set_format("torch")

Map: 100%|██████████| 1725/1725 [00:00<00:00, 73078.88 examples/s]


### 数据可视化

为了让大家直观了解到本案例使用的数据，我们选取数据集中第一条数据进行展示。

In [8]:
# 查看tokenized数据集的结构
print("Tokenized数据集特征:", tokenized_datasets['train'].features)

# 查看第一条处理后的数据
example = tokenized_datasets['train'][0]
print("\n处理后的第一条数据:")
for key, value in example.items():
    print(f"{key}: {value}")
    
    # 如果是input_ids，可以解码查看文本内容
    if key == 'input_ids':
        decoded_text = tokenizer.decode(value, skip_special_tokens=False)
        print(f"解码后的文本: {decoded_text}")

Tokenized数据集特征: {'label_text': Value('string'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': Value('int64')}

处理后的第一条数据:
label_text: equivalent
input_ids: [  6091    299   8345  13185    806  10641   1154   8711    566   2598
    330    279  11298    330   1154    315  35092   1582  51472    806
   5904    659   3945  14443    311   1435    438   1172    330    279
  11298    330   1154   3303    299   8345  13185    806  10641    315
  35092   1582  51472    806   5904    659 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151667 151667 151667 151667 151667 151667 151667
 151667 151667 151667 151

## 模型构建

加载预训练模型并进行微调

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(args.model_name_or_path, num_labels=2)

model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(model.config.vocab_size + num_added_toks)

[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(151939, 896)

给出的例子中微调算法采用IA3, 该算法通过学习向量来对激活层加权进行缩放，从而获得更强的性能，同时仅引入相对少量的新参数。

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

if args.is_lora:
    # build peft model
    peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=True)
    model = get_peft_model(model, peft_config)
    # print(model)
    model.print_trainable_parameters()

/usr/local/python3.10.14/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2276: UserWarning: fan_in_fan_out is set to True but the target module is `torch.nn.Linear`. Setting fan_in_fan_out to False.
  warnings.warn(


trainable params: 542,464 || all params: 494,579,712 || trainable%: 0.1097


## 模型训练与推理

In [11]:
from transformers import Trainer, TrainingArguments
import evaluate

定义TrainingArguments

In [12]:
if args.debug:
    args.num_train_epochs = 2

training_args = TrainingArguments(
    output_dir=args.output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=args.learning_rate,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    num_train_epochs=args.num_train_epochs,
    weight_decay=args.weight_decay,
)

加载评估指标（对于 MRPC 任务，通常关注准确率和 F1 值）

In [13]:
# 需要安装scikit-learn
! pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple


In [14]:
metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

定义Trainer

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

Detected kernel version 4.19.90, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


训练与推理

In [16]:
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.401500,1.227675,0.735294,0.830721
2,1.126700,1.081169,0.786765,0.852292


/usr/local/python3.10.14/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/python3.10.14/lib/python3.10/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=7336, training_loss=1.4378796684572974, metrics={'train_runtime': 3880.2673, 'train_samples_per_second': 1.891, 'train_steps_per_second': 1.891, 'total_flos': 4038963013091328.0, 'train_loss': 1.4378796684572974, 'epoch': 2.0})